# MangoLeaf AI: paired threshold fitting v3.2

Use the **two new models** from the background-replacement training v3.1 notebook.
This workflow reuses the validation-based method from threshold notebook v3.1; the
new revision adds the paired training-bundle contract and comparison reporting.
It does not retrain, recalibrate, or modify either model.

1. Audit the shared images and both model bundles.
2. Fit each model's margin and entropy on validation mango + OOD tune only.
3. Save both settings, including any infeasible result, before evaluation.
4. Evaluate fixed settings on test mango + OOD report; do not retune afterward.

Confidence stays at 0.70. Minimum validation retention stays at 95%, which means
accepted images, not necessarily correctly diagnosed images. A model that cannot
meet this target gets an explicit infeasible outcome, not silently relaxed thresholds.
Each feasible model also gets a confidence-only reference so you can see the extra
effect of margin/entropy. That reference is not the deployed historical v3 model.

The original test/OOD report data were used in earlier development. These results
are a comparison on reused data, not an independent field-test claim. Do not use
report failures to retune and then present the same report as an untouched test.

## Uploads and paths

Reuse the **whole training run folder** printed by the v3.1 notebook, including
`control/`, `background/`, `input_manifest.json`, `mask_review.json`, `plan.json`, and
`training_complete.json`. Keep the dataset, replacement backgrounds, and OOD files
in Drive. No original v3 resume bundle or old `.npy` files are needed.

Set `TRAINING_RUN` below to the exact completed run. This notebook reads extracted
folders, not ZIP archives. It checks the original split by relative filename,
class, and content hash. Moving a complete input folder is allowed; changing its
images is not. Runtime versions must match training for a controlled comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv, hashlib, json, math, platform, random, subprocess, sys, uuid, zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps, ImageDraw, __version__ as pillow_version

CLASS_NAMES = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker',
               'healthy', 'mango_scab', 'sooty_mold']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
IMAGE_SIZE = (224, 224)
SEED = 42
BATCH_SIZE = 32
CONFIDENCE_THRESHOLD = 0.70
RUNTIME = {'python': platform.python_version(), 'tensorflow': tf.__version__,
           'keras': tf.keras.__version__, 'numpy': np.__version__, 'pillow': pillow_version,
           'devices': [str(d) for d in tf.config.list_physical_devices()]}
tf.keras.utils.set_random_seed(SEED)
tf.keras.mixed_precision.set_global_policy('float32')
tf.config.experimental.enable_op_determinism()
print(RUNTIME)

TRAINING_RUN = Path('/content/drive/MyDrive/mangoleaf_background_v3_1/PASTE_RUN_FOLDER_HERE')
DATASET_DIR = Path('/content/drive/MyDrive/mangoleaf_v3/dataset_split/dataset_split')
OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mangoleaf_threshold_v3_2')
OOD_EXCLUDE = []  # Relative paths, only after reviewing whether an image is in scope.
MIN_ID_RETENTION = 0.95
GRID = [i / 100 for i in range(101)]
ARMS = ['control', 'background']
EVALUATION_STATUS = 'previously_used'
EVALUATION_NOTES = 'Original v3 test/report images influenced earlier model and threshold development.'
CONFIDENCE_ONLY = {'confidenceThreshold': 0.70, 'marginThreshold': 0.0, 'entropyThreshold': 1.0}
RUN_DIR = OUTPUT_ROOT / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_once(path, value):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.write('\n')


def list_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f'Image folder missing: {folder}')
    paths = sorted((p for p in folder.rglob('*')
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS),
                   key=lambda p: p.relative_to(folder).as_posix())
    if not paths:
        raise ValueError(f'No supported images in {folder}')
    return paths


def mango_records(folder, subset):
    directories = {p.name for p in folder.iterdir() if p.is_dir() and not p.name.startswith('.')}
    if directories != set(CLASS_NAMES):
        raise ValueError(f'{folder} must contain exactly these class folders: {CLASS_NAMES}')
    if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in folder.iterdir()):
        raise ValueError(f'Move no files automatically: images in {folder} need class subfolders.')
    rows = []
    for label in CLASS_NAMES:
        for path in list_images(folder / label):
            rows.append({'subset': subset, 'path': str(path.resolve()),
                         'relativePath': path.relative_to(folder).as_posix(), 'label': label})
    return rows


def audit_records(rows):
    by_hash = {}
    same_subset_duplicates = []
    for row in rows:
        path = Path(row['path'])
        try:
            with Image.open(path) as picture:
                picture.convert('RGB').load()
        except Exception as error:
            raise ValueError(f'Unreadable image: {path}') from error
        row['sha256'] = sha256_file(path)
        previous = by_hash.get(row['sha256'])
        if previous:
            if (previous['subset'], previous['label']) != (row['subset'], row['label']):
                raise ValueError('Identical image crosses subsets/labels: '
                                 f"{previous['path']} and {row['path']}")
            same_subset_duplicates.append([previous['path'], row['path']])
        else:
            by_hash[row['sha256']] = row
    return same_subset_duplicates


def verify_records_unchanged(rows):
    for row in rows:
        if sha256_file(row['path']) != row['sha256']:
            raise RuntimeError(f"Image changed during this run: {row['path']}")

In [ ]:
def disjoint_output(output, inputs):
    a = Path(output).resolve()
    for item in inputs:
        b = Path(item).resolve()
        if a == b or a in b.parents or b in a.parents:
            raise ValueError(f'Output/input trees overlap: {a} and {b}')

def verify_bundle_inputs(manifest, current):
    for subset in ['train', 'val', 'test']:
        before = {(r['relativePath'], r['sha256'], r['label']) for r in manifest['images'] if r['subset'] == subset}
        now = {(r['relativePath'], r['sha256'], r['label']) for r in current if r['subset'] == subset}
        if before != now:
            raise ValueError(f'{subset} differs from the training snapshot; do not mix datasets.')

def paired_case_rows(first, second):
    left = {r['relativePath']: r for r in first}
    right = {r['relativePath']: r for r in second}
    if left.keys() != right.keys():
        raise ValueError('Comparison arms have different evaluation images.')
    return [{'relativePath': k, 'label': left[k]['label'],
             'controlClass': left[k]['returnedClass'], 'backgroundClass': right[k]['returnedClass'],
             'controlAccepted': left[k]['accepted'], 'backgroundAccepted': right[k]['accepted'],
             'controlCorrect': left[k]['acceptedCorrect'], 'backgroundCorrect': right[k]['acceptedCorrect']}
            for k in sorted(left)]

def classification_metrics(rows):
    matrix = [[0] * len(CLASS_NAMES) for _ in CLASS_NAMES]
    for row in rows:
        matrix[CLASS_NAMES.index(row['label'])][row['predictedIndex']] += 1
    per_class = {}
    for index, label in enumerate(CLASS_NAMES):
        true_positive = matrix[index][index]
        support = sum(matrix[index])
        predicted = sum(row[index] for row in matrix)
        precision = true_positive / predicted if predicted else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[label] = {'precision': precision, 'recall': recall, 'f1': f1, 'support': support}
    return {'accuracy': sum(matrix[i][i] for i in range(len(CLASS_NAMES))) / len(rows),
            'macroF1': sum(r['f1'] for r in per_class.values()) / len(CLASS_NAMES),
            'perClass': per_class, 'confusion': {'rows': CLASS_NAMES, 'columns': CLASS_NAMES, 'counts': matrix},
            'zeroDivisionPolicy': 'Undefined precision/recall/F1 are reported as zero; support is shown.'}

In [ ]:
def probability_stats(probabilities):
    stats = []
    for row in probabilities:
        values = [float(p) for p in row]
        if len(values) != len(CLASS_NAMES) or any(
                not math.isfinite(p) or p < 0 or p > 1 for p in values):
            raise ValueError('Expected six finite probabilities in [0, 1].')
        if not math.isclose(sum(values), 1.0, abs_tol=1e-4):
            raise ValueError('Expected softmax probabilities summing to one.')
        ordered = sorted(values, reverse=True)
        stats.append({'confidence': ordered[0], 'margin': ordered[0] - ordered[1],
                      'entropy': -sum(p * math.log(p) for p in values if p > 0)
                                 / math.log(len(values)),
                      'predictedIndex': values.index(ordered[0])})
    return stats


def accepted(stat, thresholds):
    return (stat['confidence'] >= thresholds['confidenceThreshold']
            and stat['margin'] >= thresholds['marginThreshold']
            and stat['entropy'] <= thresholds['entropyThreshold'])


def select_thresholds(val_stats, ood_tune_stats, grid, confidence, min_retention):
    # Only tuning data enters this function. Counts avoid rounded-score ties.
    if not val_stats or not ood_tune_stats:
        raise ValueError('Both tuning subsets must contain predictions.')
    if not 0 < min_retention <= 1 or not 0 <= confidence <= 1:
        raise ValueError('Invalid confidence or retention target.')
    if not grid or any(not math.isfinite(x) or not 0 <= x <= 1 for x in grid):
        raise ValueError('Grid must contain finite values between zero and one.')
    required = math.ceil(min_retention * len(val_stats) - 1e-9)
    best, best_key, rows = None, None, []
    for margin in grid:
        val_eligible = [s for s in val_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        ood_eligible = [s for s in ood_tune_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        for entropy in grid:
            kept = sum(s['entropy'] <= entropy for s in val_eligible)
            rejected = len(ood_tune_stats) - sum(s['entropy'] <= entropy for s in ood_eligible)
            feasible = kept >= required
            rows.append({'margin': margin, 'entropy': entropy, 'valAccepted': kept,
                         'valTotal': len(val_stats), 'oodRejected': rejected,
                         'oodTotal': len(ood_tune_stats), 'meetsRetentionTarget': feasible})
            # Ties: more mango retention, then lower margin, then higher entropy.
            key = (rejected, kept, -margin, entropy)
            if feasible and (best_key is None or key > best_key):
                best_key = key
                best = {'confidenceThreshold': confidence, 'marginThreshold': margin,
                        'entropyThreshold': entropy}
    if best is None:
        raise ValueError('No grid pair meets the validation retention target. '
                         'Review tuning data or the prespecified target; '
                         'no replacement thresholds were invented.')
    return best, rows


def scored_rows(records, stats, thresholds):
    if len(records) != len(stats):
        raise ValueError('Image and prediction counts differ.')
    rows = []
    for record, stat in zip(records, stats):
        keep = accepted(stat, thresholds)
        prediction = CLASS_NAMES[stat['predictedIndex']]
        reasons = []
        if stat['confidence'] < thresholds['confidenceThreshold']: reasons.append('confidence')
        if stat['margin'] < thresholds['marginThreshold']: reasons.append('margin')
        if stat['entropy'] > thresholds['entropyThreshold']: reasons.append('entropy')
        rows.append({**record, **stat, 'predictedClass': prediction, 'accepted': keep,
                     'returnedClass': prediction if keep else 'uncertain',
                     'rejectionReasons': ','.join(reasons),
                     'acceptedCorrect': keep and prediction == record['label']})
    return rows


def summarize(rows):
    if not rows:
        raise ValueError('Cannot report an empty subset.')
    count = len(rows)
    kept = sum(row['accepted'] for row in rows)
    report = {'total': count, 'accepted': kept, 'rejected': count - kept,
              'retention': kept / count, 'rejectionRate': (count - kept) / count}
    if all(row['label'] == 'OOD' for row in rows):
        report['falseAcceptRate'] = kept / count
        return report
    if any(row['label'] == 'OOD' for row in rows):
        raise ValueError('Report mango and OOD subsets separately.')
    correct = sum(row['acceptedCorrect'] for row in rows)
    report.update({'rawAccuracy': sum(row['predictedClass'] == row['label'] for row in rows) / count,
                   'acceptedCorrect': correct, 'acceptedWrong': kept - correct,
                   'acceptedAccuracy': correct / kept if kept else None,
                   'acceptedErrorsPerInput': (kept - correct) / count})
    per_class = {}
    confusion = [[0] * (len(CLASS_NAMES) + 1) for _ in CLASS_NAMES]
    for row in rows:
        column = row['predictedIndex'] if row['accepted'] else len(CLASS_NAMES)
        confusion[CLASS_NAMES.index(row['label'])][column] += 1
    for label in CLASS_NAMES:
        group = [row for row in rows if row['label'] == label]
        class_kept = sum(row['accepted'] for row in group)
        class_correct = sum(row['acceptedCorrect'] for row in group)
        per_class[label] = {'total': len(group), 'acceptedCorrect': class_correct,
                            'acceptedWrong': class_kept - class_correct,
                            'rejected': len(group) - class_kept}
    report['perClass'] = per_class
    report['confusion'] = {'rows': CLASS_NAMES, 'columns': CLASS_NAMES + ['uncertain'], 'counts': confusion}
    return report


def write_csv_once(path, rows):
    if not rows:
        raise ValueError('Cannot write an empty report.')
    with Path(path).open('x', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def read_frozen_selection(path, expected_sha256):
    if sha256_file(path) != expected_sha256:
        raise RuntimeError('Saved selection changed. Start a new run; do not retune during evaluation.')
    return json.loads(Path(path).read_text(encoding='utf-8'))

In [ ]:
disjoint_output(OUTPUT_ROOT, [TRAINING_RUN, DATASET_DIR, OOD_DIR])
if EVALUATION_STATUS != 'previously_used':
    raise ValueError('This paired notebook reproduces the historical split. Fresh evaluation needs a separately documented protocol.')
training_manifest_path = TRAINING_RUN / 'input_manifest.json'
training_review_path = TRAINING_RUN / 'mask_review.json'
completed = json.loads((TRAINING_RUN / 'training_complete.json').read_text(encoding='utf-8'))
training_manifest = json.loads(training_manifest_path.read_text(encoding='utf-8'))
if sha256_file(training_manifest_path) != completed['inputManifestSha256']:
    raise ValueError('Training manifest has changed since model export.')
if sha256_file(training_review_path) != completed['maskReviewSha256']:
    raise ValueError('Training mask review has changed since model export.')
if sha256_file(TRAINING_RUN / 'plan.json') != completed['planSha256']:
    raise ValueError('Training plan has changed since model export.')
groups = {s: mango_records(DATASET_DIR / s, s) for s in ['train', 'val', 'test']}
audit_records(sum(groups.values(), []))
verify_bundle_inputs(training_manifest, sum(groups.values(), []))
val_records, test_records = groups['val'], groups['test']

tune_dir, report_dir = OOD_DIR / 'tune', OOD_DIR / 'report'
if tune_dir.is_dir() != report_dir.is_dir():
    raise ValueError('OOD layout needs both tune/ and report/, or neither.')
if tune_dir.is_dir():
    tune_paths, report_paths = list_images(tune_dir), list_images(report_dir)
    if set(list_images(OOD_DIR)) != set(tune_paths + report_paths):
        raise ValueError('OOD images found outside tune/report folders.')
    split_method = 'existing tune/report folders'
else:
    paths = list_images(OOD_DIR)
    order = np.random.RandomState(SEED).permutation(len(paths))
    half = len(paths) // 2
    tune_paths = [paths[i] for i in sorted(order[:half])]
    report_paths = [paths[i] for i in sorted(order[half:])]
    split_method = 'sorted relative paths, NumPy RandomState seed 42, 50/50'
available = {p.relative_to(OOD_DIR).as_posix() for p in tune_paths + report_paths}
if set(OOD_EXCLUDE) - available:
    raise ValueError('An OOD exclusion path does not exist.')
def ood_rows(paths, subset):
    return [{'subset': subset, 'label': 'OOD', 'path': str(p.resolve()),
             'relativePath': p.relative_to(OOD_DIR).as_posix()}
            for p in paths if p.relative_to(OOD_DIR).as_posix() not in OOD_EXCLUDE]
tune_records = ood_rows(tune_paths, 'ood_tune')
report_records = ood_rows(report_paths, 'ood_report')
if not tune_records or not report_records:
    raise ValueError('Both OOD subsets need images after exclusions.')
all_records = sum(groups.values(), []) + tune_records + report_records
if audit_records(all_records):
    raise ValueError('Exact duplicate evaluation images require review before continuing.')
training_exposure = {r['sha256'] for r in training_manifest['images'] if r['subset'] in {'train', 'background'}}
if any(r['sha256'] in training_exposure for r in tune_records + report_records):
    raise ValueError('OOD images overlap training images or replacement backgrounds.')
original_ood = {r['sha256'] for r in training_manifest['images'] if r['subset'] == 'reserved_ood'}
current_ood = {sha256_file(p) for p in tune_paths + report_paths}
if current_ood != original_ood:
    raise ValueError('OOD collection changed since the experiment was planned; use a separately documented evaluation.')

model_infos = {}
for arm in ARMS:
    model_path = TRAINING_RUN / arm / 'model_calibrated.keras'
    info_path = TRAINING_RUN / arm / 'resume_info.json'
    info = json.loads(info_path.read_text(encoding='utf-8'))
    if info != completed['arms'][arm]:
        raise ValueError(f'{arm} metadata differs from the completed training record.')
    if info['arm'] != arm or info['trial'] != '3.1' or info['classes'] != CLASS_NAMES:
        raise ValueError(f'{arm}: incompatible training bundle.')
    if info['inputManifestSha256'] != completed['inputManifestSha256'] or info['maskReviewSha256'] != completed['maskReviewSha256']:
        raise ValueError(f'{arm}: provenance belongs to a different run.')
    if info['planSha256'] != completed['planSha256']:
        raise ValueError(f'{arm}: training plan belongs to a different run.')
    if info['modelSha256'] != sha256_file(model_path):
        raise ValueError(f'{arm}: calibrated model hash does not match metadata.')
    if info['temperatureFoldedIntoWeights'] is not True or info['confidenceThreshold'] != 0.70:
        raise ValueError('Expected calibrated weights and the fixed 0.70 threshold.')
    if not math.isfinite(float(info['temperature'])) or info['temperature'] <= 0:
        raise ValueError('Invalid saved temperature.')
    for library in ['tensorflow', 'keras', 'numpy', 'pillow']:
        if info['runtime'][library] != RUNTIME[library]:
            raise RuntimeError(f'{library} differs from training. Use a matching runtime before comparing models.')
    model_infos[arm] = {**info, 'modelPath': str(model_path), 'infoSha256': sha256_file(info_path)}
if model_infos['control']['initialWeightsSha256'] != model_infos['background']['initialWeightsSha256']:
    raise ValueError('The paired models did not share initial weights.')
RUN_DIR.mkdir(parents=True, exist_ok=False)
MANIFEST_PATH = RUN_DIR / 'input_manifest.json'
write_json_once(MANIFEST_PATH, {'trainingRun': str(TRAINING_RUN), 'images': all_records,
    'trainingManifestSha256': completed['inputManifestSha256'], 'oodSplitMethod': split_method,
    'oodExcluded': OOD_EXCLUDE, 'evaluationStatus': EVALUATION_STATUS, 'evaluationNotes': EVALUATION_NOTES})
MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
for subset in ['val', 'test', 'ood_tune', 'ood_report']:
    print(subset, sum(r['subset'] == subset for r in all_records))
print('Paired results will go to:', RUN_DIR)

In [ ]:
def load_arm(arm):
    info = model_infos[arm]
    if sha256_file(info['modelPath']) != info['modelSha256']:
        raise RuntimeError('Model changed after audit: ' + arm)
    if sha256_file(TRAINING_RUN / arm / 'resume_info.json') != info['infoSha256']:
        raise RuntimeError('Model metadata changed after audit: ' + arm)
    model = tf.keras.models.load_model(info['modelPath'], compile=False, safe_mode=True)
    if tuple(model.input_shape[1:]) != (224, 224, 3) or tuple(model.output_shape[1:]) != (6,):
        raise ValueError('Unexpected model input/output shape.')
    return model


def predict_records(model, records):
    predictions = []
    for start in range(0, len(records), BATCH_SIZE):
        tensors = []
        for row in records[start:start + BATCH_SIZE]:
            if sha256_file(row['path']) != row['sha256']:
                raise RuntimeError('Image changed after audit: ' + row['path'])
            with Image.open(row['path']) as picture:
                pixels = np.asarray(picture.convert('RGB'), dtype=np.float32)
            tensors.append(tf.image.resize(pixels, IMAGE_SIZE, method='bilinear', antialias=False))
        predictions.extend(model(tf.stack(tensors), training=False).numpy().tolist())
    if len(predictions) != len(records):
        raise RuntimeError('Prediction count mismatch.')
    return predictions

## Fit both arms, then freeze both selections

Only validation and OOD tune predictions enter this cell. Each arm maximizes OOD tune
rejections while retaining at least 95% of mango validation images. Ties prefer more
retained mango images, then a lower margin, then a higher entropy limit. Zero margin
is allowed: at confidence >= 0.70, margin >= 0.40 already follows mathematically for
normalized probabilities, apart from rounding.

An infeasible arm is recorded and receives confidence-only evaluation later. It does
not receive an invented candidate configuration or a lower retention target.

In [ ]:
selections = {}
for arm in ARMS:
    arm_dir = RUN_DIR / arm
    arm_dir.mkdir(exist_ok=False)
    model = load_arm(arm)
    val_stats = probability_stats(predict_records(model, val_records))
    tune_stats = probability_stats(predict_records(model, tune_records))
    try:
        thresholds, search = select_thresholds(val_stats, tune_stats, GRID, CONFIDENCE_THRESHOLD, MIN_ID_RETENTION)
        outcome = {'status': 'feasible', 'thresholds': thresholds}
        write_csv_once(arm_dir / 'threshold_search.csv', search)
        outcome['validation'] = summarize(scored_rows(val_records, val_stats, thresholds))
        outcome['oodTune'] = summarize(scored_rows(tune_records, tune_stats, thresholds))
    except ValueError as error:
        if not str(error).startswith('No grid pair'): raise
        outcome = {'status': 'infeasible', 'thresholds': None, 'reason': str(error)}
    write_csv_once(arm_dir / 'tuning_predictions.csv',
        scored_rows(val_records, val_stats, CONFIDENCE_ONLY) + scored_rows(tune_records, tune_stats, CONFIDENCE_ONLY))
    selections[arm] = {**outcome, 'modelSha256': model_infos[arm]['modelSha256'],
                       'infoSha256': model_infos[arm]['infoSha256']}
    print(arm, outcome)
    del model
SELECTION_PATH = RUN_DIR / 'selected_thresholds.json'
write_json_once(SELECTION_PATH, {'arms': selections, 'inputManifestSha256': MANIFEST_SHA256,
    'confidence': CONFIDENCE_THRESHOLD, 'minimumValidationRetention': MIN_ID_RETENTION,
    'gridStep': 0.01, 'runtime': RUNTIME, 'savedBeforeEvaluation': True,
    'savedAt': datetime.now(timezone.utc).isoformat()})
SELECTION_SHA256 = sha256_file(SELECTION_PATH)
print('BOTH SELECTIONS FROZEN:', SELECTION_PATH)

## Evaluate saved settings

This cell cannot change either threshold selection. It reports raw classification
accuracy, accepted correct/wrong diagnoses, rejected mango images, per-class counts,
confusion with a rejection column, and OOD false accepts. The paired CSV identifies
which individual cases improved or worsened. A confident wrong result and a rejected
valid mango image are different failures; do not combine them into one accuracy claim.

In [ ]:
frozen = read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
if sha256_file(MANIFEST_PATH) != frozen['inputManifestSha256']:
    raise RuntimeError('Input manifest changed after threshold selection.')
verify_records_unchanged(all_records)
evaluations, candidate_rows = {}, {}
for arm in ARMS:
    selected = frozen['arms'][arm]
    if selected['modelSha256'] != model_infos[arm]['modelSha256'] or selected['infoSha256'] != model_infos[arm]['infoSha256']:
        raise RuntimeError('Frozen selection belongs to different weights/metadata.')
    model = load_arm(arm)
    test_stats = probability_stats(predict_records(model, test_records))
    report_stats = probability_stats(predict_records(model, report_records))
    references = scored_rows(test_records, test_stats, CONFIDENCE_ONLY) + scored_rows(report_records, report_stats, CONFIDENCE_ONLY)
    write_csv_once(RUN_DIR / arm / 'confidence_only_predictions.csv', references)
    evaluation = {'status': selected['status'], 'thresholds': selected['thresholds'],
        'rawClassification': classification_metrics(references[:len(test_records)]),
        'confidenceOnly': {'mangoTest': summarize(references[:len(test_records)]),
                           'oodReport': summarize(references[len(test_records):])}}
    write_csv_once(RUN_DIR / arm / 'per_class_metrics.csv', [
        {'class': label, **metrics} for label, metrics in evaluation['rawClassification']['perClass'].items()])
    if selected['status'] == 'feasible':
        rows = scored_rows(test_records, test_stats, selected['thresholds']) + scored_rows(report_records, report_stats, selected['thresholds'])
        candidate_rows[arm] = rows
        write_csv_once(RUN_DIR / arm / 'candidate_predictions.csv', rows)
        evaluation['candidate'] = {'mangoTest': summarize(rows[:len(test_records)]),
                                    'oodReport': summarize(rows[len(test_records):])}
    evaluations[arm] = evaluation
    del model
paired = []
if len(candidate_rows) == 2:
    # Subsets prefix the keys so a mango and an OOD file cannot collide by filename.
    left = [{**r, 'relativePath': r['subset'] + '/' + r['relativePath']} for r in candidate_rows['control']]
    right = [{**r, 'relativePath': r['subset'] + '/' + r['relativePath']} for r in candidate_rows['background']]
    paired = paired_case_rows(left, right)
    write_csv_once(RUN_DIR / 'paired_cases.csv', paired)
read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
write_json_once(RUN_DIR / 'evaluation.json', {'arms': evaluations,
    'selectionSha256': SELECTION_SHA256, 'evaluationStatus': EVALUATION_STATUS,
    'evaluationNotes': EVALUATION_NOTES, 'browserParityMeasured': False,
    'finishedAt': datetime.now(timezone.utc).isoformat()})
print(json.dumps(evaluations, indent=2))

## Export evidence, without deployment

The ZIP contains reports/configurations, not model weights or input images. The paired
calibrated `.keras` files stay in the training run. Each feasible arm gets a config
matched by model hash; never combine one arm's weights with the other's thresholds.
The background candidate has model label `model-v3.1.0`; `v3.2` here is the fitting
notebook revision, not another retrained model. No app version or deployed file changes.

In [ ]:
for arm in ARMS:
    selected = frozen['arms'][arm]
    if selected['status'] != 'feasible': continue
    info = model_infos[arm]
    write_json_once(RUN_DIR / arm / 'model_config.json', {
        'modelVersion': info['modelVersion'], 'trial': '3.1', 'classes': CLASS_NAMES,
        'inputSize': [224, 224], 'pixelRange': '0-255 raw, MobileNetV3 rescales internally',
        'temperature': info['temperature'], 'temperatureFoldedIntoWeights': True,
        'thresholds': selected['thresholds'],
        'provenance': {'arm': arm, 'modelSha256': info['modelSha256'],
            'selectionSha256': SELECTION_SHA256, 'trainingRun': str(TRAINING_RUN),
            'evaluationStatus': EVALUATION_STATUS, 'browserParityMeasured': False}})
lines = ['# Background replacement experiment: paired evaluation', '',
    'These are exploratory results on previously used evaluation data.',
    'Both threshold selections were saved before test/report inference.', '',
    '| Arm | Status | Correct accepted | Wrong accepted | Mango rejected | OOD accepted |',
    '| --- | --- | ---: | ---: | ---: | ---: |']
for arm in ARMS:
    result = evaluations[arm]
    if result['status'] != 'feasible':
        lines.append(f'| {arm} | infeasible | — | — | — | — |')
    else:
        mango, ood = result['candidate']['mangoTest'], result['candidate']['oodReport']
        lines.append(f"| {arm} | feasible | {mango['acceptedCorrect']}/{mango['total']} | {mango['acceptedWrong']} | {mango['rejected']} | {ood['accepted']}/{ood['total']} |")
lines += ['', 'Inspect paired_cases.csv and per-class errors, not only accepted accuracy.',
    'Neither model is automatically selected for deployment.',
    'One seed, reused test/report images, synthetic edges, and mask-review coverage limit conclusions.',
    'Background replacement does not explicitly teach a non-mango class.',
    'Fresh field evaluation and TensorFlow.js/browser parity are still required before adoption.']
with (RUN_DIR / 'REPORT.md').open('x', encoding='utf-8') as handle:
    handle.write('\n'.join(lines) + '\n')
archive_path = RUN_DIR.with_suffix('.zip')
with zipfile.ZipFile(archive_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file(): archive.write(path, path.relative_to(RUN_DIR).as_posix())
print('\n'.join(lines))
print('Evidence ZIP:', archive_path)
from google.colab import files
files.download(str(archive_path))

## References and record keeping

Save completed output separately as `MangoLeaf_AI_Threshold_Fitting_v3_2_Results.ipynb`.
Keep the clean template and older notebooks unchanged. This notebook follows the
validation/tuning versus test/report separation documented in the repository's v3.1
threshold guide. No float16 simulation is used as a substitute for actual browser tests.

[Model selection and held-out evaluation](https://scikit-learn.org/stable/modules/cross_validation.html),
[background dependence](https://arxiv.org/abs/2006.09994),
[Outlier Exposure, a separate approach to rejection](https://arxiv.org/abs/1812.04606).